# Bloque 1 — Demo: Técnicas de Big Data

Este notebook acompaña la presentación del Bloque 1 y demuestra en vivo las técnicas de análisis sin IA:
análisis de red, series temporales y TF-IDF.

Usamos el dataset de **Carding Forums** como ejemplo porque es pequeño y limpio — perfecto para demos.

> **Prerrequisito**: haber ejecutado `CardingForums/01_ingenieria_datos.ipynb` para tener los archivos `.parquet` limpios.

## Imports

Cargamos las librerías que vamos a usar.

- **pandas**: para manipular tablas de datos
- **networkx**: para construir y analizar grafos (redes de usuarios)
- **plotly**: para visualizaciones interactivas
- **sklearn**: para TF-IDF (análisis de texto)
- **community**: algoritmo de Louvain para detección de comunidades

In [ ]:
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer

try:
    import community as community_louvain
    HAS_LOUVAIN = True
except ImportError:
    HAS_LOUVAIN = False
    print('python-louvain no instalado — se usará detección básica de NetworkX')

DATA_DIR = Path('../results')
print('Directorio de datos:', DATA_DIR.resolve())

## 1. Cargar los datos limpios

Cargamos los archivos `.parquet` generados en el notebook de ingeniería de datos.

Un archivo **Parquet** es un formato de almacenamiento columnar muy eficiente — mucho más rápido de cargar que un CSV y ocupa menos espacio en disco. Pandas lo lee directamente con `pd.read_parquet()`.

In [ ]:
posts = pd.read_parquet(DATA_DIR / '01_posts_clean.parquet')
users = pd.read_parquet(DATA_DIR / '01_users_clean.parquet')

print(f'Posts cargados:   {len(posts):,}')
print(f'Usuarios cargados: {len(users):,}')
posts.head(3)

## 2. Análisis de red — ¿quién habla con quién?

### ¿Qué es un grafo de co-participación?

Dos usuarios están conectados si ambos han posteado en el mismo hilo de discusión.
La idea es que participar en el mismo hilo implica algún nivel de interacción o conocimiento mutuo.

Construimos el grafo así:
1. Para cada hilo, obtenemos la lista de usuarios que postearon en él
2. Conectamos todos los pares de usuarios de esa lista entre sí
3. Repetimos para todos los hilos

El resultado es un **grafo no dirigido** donde cada nodo es un usuario y cada arista representa co-participación en al menos un hilo.

Limitamos a los primeros 5.000 hilos para que el grafo sea manejable en RAM.

In [ ]:
thread_col = 'topic_id' if 'topic_id' in posts.columns else 'threadid'
user_col   = 'userid'

posts = posts.dropna(subset=[thread_col, user_col])

# Tomar los primeros 5000 hilos únicos
top_threads = posts[thread_col].value_counts().head(5000).index
df = posts[posts[thread_col].isin(top_threads)]

# Construir el grafo
G = nx.Graph()
for thread_id, group in df.groupby(thread_col):
    users_in_thread = group[user_col].unique().tolist()
    for i in range(len(users_in_thread)):
        for j in range(i + 1, len(users_in_thread)):
            u, v = str(users_in_thread[i]), str(users_in_thread[j])
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1  # más hilos compartidos = arista más fuerte
            else:
                G.add_edge(u, v, weight=1)

print(f'Nodos (usuarios): {G.number_of_nodes():,}')
print(f'Aristas (co-participaciones): {G.number_of_edges():,}')

### Métricas de centralidad

**Degree centrality**: ¿con cuántos usuarios distintos ha coincidido este usuario? Un degree alto significa que el usuario es muy activo y alcanza a muchos otros.

**Betweenness centrality**: ¿cuántas rutas cortas entre otros pares de usuarios pasan por este nodo? Un valor alto indica que este usuario actúa de **puente** entre subcomunidades — si lo eliminas del grafo, grupos que antes estaban conectados quedarían separados. Estos son los actores más interesantes para inteligencia.

In [ ]:
# Trabajamos sobre el componente gigante (el subgrafo más grande)
giant = G.subgraph(max(nx.connected_components(G), key=len)).copy()
print(f'Componente gigante: {giant.number_of_nodes():,} nodos ({giant.number_of_nodes()/G.number_of_nodes()*100:.1f}% del total)')

degree     = nx.degree_centrality(giant)
# Betweenness sobre muestra de 500 nodos para ir rápido
betweenness = nx.betweenness_centrality(giant, k=min(500, giant.number_of_nodes()), normalized=True)

# Top 10 por betweenness
uid_to_name = dict(zip(users['userid'].astype(str), users.get('username', users.get('name', users.index.astype(str)))))
top_bw = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]
print('\nTop 10 brokers (betweenness):')
for uid, bw in top_bw:
    name = uid_to_name.get(uid, uid)
    print(f'  {name:<25} {bw:.4f}')

### Visualización interactiva — top 150 nodos

Visualizamos los 150 nodos con mayor degree. El **tamaño** del nodo representa su degree (más conexiones = más grande). El **color** representa la comunidad detectada por el algoritmo de Louvain.

Pasa el ratón sobre un nodo para ver el nombre del usuario.

In [ ]:
import numpy as np

# Seleccionar top 150 nodos por degree
top150 = sorted(degree.items(), key=lambda x: x[1], reverse=True)[:150]
top150_ids = [uid for uid, _ in top150]
sub = giant.subgraph(top150_ids).copy()

# Detectar comunidades
if HAS_LOUVAIN:
    partition = community_louvain.best_partition(sub)
else:
    partition = {n: 0 for n in sub.nodes()}

# Layout del grafo
pos = nx.spring_layout(sub, seed=42, k=1.5)

edge_x, edge_y = [], []
for u, v in sub.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

node_x = [pos[n][0] for n in sub.nodes()]
node_y = [pos[n][1] for n in sub.nodes()]
node_names = [uid_to_name.get(n, n) for n in sub.nodes()]
node_colors = [partition.get(n, 0) for n in sub.nodes()]
node_sizes  = [degree.get(n, 0) * 300 + 5 for n in sub.nodes()]

fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
                          line=dict(width=0.3, color='#888'), hoverinfo='none'))
fig.add_trace(go.Scatter(x=node_x, y=node_y, mode='markers+text',
                          marker=dict(size=node_sizes, color=node_colors, colorscale='Viridis',
                                      showscale=True, colorbar=dict(title='Comunidad')),
                          text=node_names, textposition='top center',
                          hovertemplate='<b>%{text}</b><extra></extra>'))
fig.update_layout(title='Red de co-participación — top 150 usuarios',
                  showlegend=False, height=700,
                  xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
fig.show()

## 3. Análisis temporal — ¿cuándo estaba activo el foro?

Los patrones temporales revelan mucho sobre una comunidad:
- **Ciclo de vida**: cuándo creció, cuándo murió
- **Horario de actividad**: inferencia de timezone (independientemente de lo que declare el usuario)
- **Eventos externos**: picos de actividad correlacionan con eventos reales

Recordatorio importante: filtramos posts con fechas anteriores a 2000 porque algunos registros tienen timestamp=0 (corrupto), que pandas interpreta como 1970-01-01.

In [ ]:
date_col = 'post_date' if 'post_date' in posts.columns else 'dateline'

df_time = posts.copy()
df_time['date'] = pd.to_datetime(df_time[date_col], errors='coerce', utc=True)
df_time = df_time[df_time['date'].dt.year >= 2000]  # Filtrar epoch-0

# Actividad mensual
monthly = df_time.set_index('date').resample('ME').size().reset_index()
monthly.columns = ['mes', 'posts']

fig = px.bar(monthly, x='mes', y='posts',
             title='Actividad mensual del foro',
             labels={'mes': 'Mes', 'posts': 'Posts'})
fig.show()

In [ ]:
# Distribución horaria — inferencia de timezone
df_time['hora'] = df_time['date'].dt.hour
hourly = df_time.groupby('hora').size().reset_index(name='posts')

fig = px.bar(hourly, x='hora', y='posts',
             title='Distribución horaria (UTC) — pico = timezone mayoritaria',
             labels={'hora': 'Hora UTC', 'posts': 'Posts'})
fig.show()

peak_hour = hourly.loc[hourly['posts'].idxmax(), 'hora']
print(f'Pico de actividad: {peak_hour}h UTC')

## 4. TF-IDF — ¿de qué habla cada subforo?

### ¿Qué es TF-IDF?

**TF** (Term Frequency): ¿cuántas veces aparece esta palabra en este subforo?
**IDF** (Inverse Document Frequency): ¿en cuántos subforos aparece esta palabra? Si aparece en todos, no es discriminante.

El resultado: palabras con **TF-IDF alto** son las que aparecen mucho en un subforo pero poco en los demás. Son las palabras más características — las que mejor describen de qué habla ese subforo específicamente.

Es como buscar lo que hace único a cada lugar, quitando las palabras comunes a todos.

In [ ]:
forum_col  = 'forum_id' if 'forum_id' in posts.columns else 'forum'
text_col   = 'post_text' if 'post_text' in posts.columns else 'pagetext'

# Agrupar texto por subforo
forum_texts = (
    posts.dropna(subset=[forum_col, text_col])
    .groupby(forum_col)[text_col]
    .apply(lambda x: ' '.join(x.astype(str).tolist()))
)

# Solo subforos con suficiente texto
forum_texts = forum_texts[forum_texts.str.len() > 500]
print(f'Subforos con suficiente texto: {len(forum_texts)}')

vectorizer = TfidfVectorizer(
    max_features=500,
    stop_words='english',
    min_df=1,
    sublinear_tf=True,   # log(1+tf) en lugar de tf — reduce el peso de palabras muy frecuentes
)
tfidf_matrix = vectorizer.fit_transform(forum_texts)
feature_names = vectorizer.get_feature_names_out()

# Top 10 términos por subforo
print('\nTop 10 términos más discriminantes por subforo:')
for i, forum_id in enumerate(forum_texts.index[:5]):  # Mostrar primeros 5 subforos
    scores = tfidf_matrix[i].toarray().flatten()
    top_idx = scores.argsort()[::-1][:10]
    top_terms = ', '.join(feature_names[top_idx])
    print(f'  Subforo {forum_id}: {top_terms}')